# A FAIR² Mental Health Survey Dataset from Kilifi County Exploration with `mlcroissant`
This notebook explores the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. We follow the MLCommons Croissant schema to load and analyze data in a standardized, reproducible, and AI-ready manner.

### Dataset Source
The dataset is defined by a Croissant schema, available at:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata and instantiate the mlcroissant Dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata
metadata = dataset.metadata.to_json()
title = dataset.metadata.name
description = dataset.metadata.description

print(f"{title}: {description}")

## 2. Data Overview
Review available record sets and fields in the dataset. All entities are referenced by their `@id` values in accordance with Croissant standards.

In [ ]:
# Examine record sets defined by the Croissant schema
record_sets = dataset.metadata.recordSet

if not record_sets:
    print("No record sets found in metadata.\nAttempting to enumerate from Croissant schema...")
    # In Croissant datasets, records may be found via hasPart or distribution for survey datasets
    # mlcroissant usually populates metadata.recordSet
    # Optionally, you might reload or inspect via dataset.records()
else:
    print(f"Found {len(record_sets)} record sets.")
    for rs in record_sets:
        print(f"RecordSet: @id={rs['@id']}")
        # Print fields referenced in the record set
        if 'field' in rs:
            fields = rs['field']
            # If single field, wrap in list
            if isinstance(fields, dict):
                fields = [fields]
            for field in fields:
                print(f"  Field: @id={field['@id']}, name={field.get('name','<no name>')}")
        else:
            print("  No fields found in this record set.")

# If record sets are missing, fall back to listing available keys
# Uncomment below to debug
# print(dataset.metadata.to_json())

## 3. Data Extraction
Load rows from a specific record set into a DataFrame. Use record set and field `@id`s for addressing.

In [ ]:
# Identify RecordSet @id(s) for extraction (replace if necessary with correct @id's from overview above)
# For demonstration, we'll assume a single record set @id is available after overview

# Example usage:
# Suppose the record set @id is 'cr:recordSet1'
# Example field @id 'cr:field:age'

# For this dataset, try to enumerate available record sets from metadata
record_sets = dataset.metadata.recordSet
record_set_ids = []
if record_sets:
    for rs in record_sets:
        record_set_ids.append(rs['@id'])

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id={record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in data for @id {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records loaded for {record_set_id}.")

# For analysis, pick first record set as example
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Example RecordSet ID for later analysis: {example_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalization, grouping, referencing field and column `@id`s. Replace the placeholder field IDs with actual ones from your dataset.

In [ ]:
# EDA for numeric and group fields
# Adjust the field @id names as discovered previously

record_set_id = example_record_set_id

# Retrieve the DataFrame
df = dataframes.get(record_set_id)

if df is not None:
    print(f"Columns: {df.columns.tolist()}")
    # Common numeric fields in mental health surveys: e.g., GAD-7, PHQ-9, PSQ scores
    # Assume fields with @id like 'gad7_score', 'phq9_score'
    numeric_field_id = None
    group_field_id = None
    # Try auto-discovering numeric and group fields
    for col in df.columns:
        if 'score' in col.lower():
            numeric_field_id = col
        if col.lower() in ['village', 'gender', 'age_group', 'group', 'level_of_education']:
            group_field_id = col
    
    # Fallback or specify numeric_field_id as needed
    if numeric_field_id is None:
        numeric_field_id = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]
    if group_field_id is None:
        group_field_id = 'village' if 'village' in df.columns else df.columns[0]
    
    print(f"Numeric field @id used: {numeric_field_id}")
    print(f"Group field @id used: {group_field_id}")
    
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields referenced by their `@id` values. For example, plot the distribution of the GAD-7 score by village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric score
if 'df' in locals() and df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Plot boxplot by group_field
    if group_field_id in df.columns:
        plt.figure(figsize=(12,7))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County with `mlcroissant`.
- We loaded metadata and records, referencing all entities by their `@id` per Croissant standards.
- We reviewed available record sets and fields, loaded data into pandas DataFrames, and performed simple EDA (filtering, normalization, grouping).
- Data distributions and relationships (e.g., GAD-7 score by village) were visualized.

This demonstrates a reproducible, FAIR approach to mental health dataset analysis in African research contexts.